# Statistics of Extracted Data on mHealth Apps

## 1. Imports and shared configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from pathlib import Path

# Region / country mappings
REGION_MAP = {
    "us": "North America", "ca": "North America", "gl":"North America",
    "mx": "Latin America", "br": "Latin America", "ar": "Latin America", 
    "cl": "Latin America",
    "de": "Europe", "fr": "Europe", "gb": "Europe", 
    "pl": "Europe",
    "tr": "Middle East", "il": "Middle East", "sa": "Middle East", 
    "ae": "Middle East",
    "in": "Asia", "jp": "Asia", "sg": "Asia", 
    "kr": "Asia",
    "za": "Africa", "mz": "Africa", "ng": "Africa", 
    "ke": "Africa",
    "au": "Oceania", "nz": "Oceania", 
    "pg": "Oceania"
}

COUNTRY_LABEL_MAP = {
    "us": "United States", "ca": "Canada", "gl":"Greenland",
    "mx": "Mexico", "br": "Brazil", "ar": "Argentina",
    "cl": "Chile",
    "de": "Germany", "fr": "France", "gb": "Great Britain",
    "pl": "Poland",
    "tr": "Turkey", "il": "Israel", "sa": "Saudi Arabia",
    "ae": "United Arab Emirates",
    "in": "India", "jp": "Japan", "sg": "Singapore",
    "kr": "South Korea",
    "za": "South Africa", "mz": "Mozambique", "ng": "Nigeria",
    "ke": "Kenya",
    "au": "Australia", "nz": "New Zealand",
    "pg": "Papua New Guinea"
}

DATA_PATH = Path('../data/mhealth_apps_metrics.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError("Could not find 'mhealth_apps_metrics.csv' in ../data.")
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

def extract_unique_item(series):
    unique_items = set()
    for items in series.dropna():
        for item in str(items).split(', '):
            item = item.strip()
            if item and item.lower() != 'nan':
                unique_items.add(item)
    return unique_items

def print_stats_block(title, stats_dict, width=40):
    print(f'\n{title}:')
    print('-' * 60)
    for stat_name, value in stats_dict.items():
        if isinstance(value, float):
            if 'Downloads' in stat_name:
                print(f'{stat_name:<{width}} {value:,.2f}')
            else:
                print(f'{stat_name:<{width}} {value:.2f}')
        else:
            print(f'{stat_name:<{width}} {value:,}')


## 2. Load and standardize the dataset

In [ ]:
result_df = pd.read_csv(DATA_PATH)

result_df['country'] = result_df['country'].astype(str).str.strip().str.lower()
result_df['region'] = result_df['country'].map(REGION_MAP)
result_df['country_label'] = result_df['country'].map(COUNTRY_LABEL_MAP)
result_df['downloads_int'] = pd.to_numeric(result_df['downloads_int'], errors='coerce')

## 3. Descriptive statistics by region and country

In [ ]:
def compute_statistics(df):
    """Compute app-level descriptive statistics within a subset."""
    unique_df = df.drop_duplicates(subset='app_id').copy()

    total_unique_apps = unique_df['app_id'].nunique()
    average_downloads = unique_df['downloads_int'].mean()
    total_downloads = unique_df['downloads_int'].sum()

    unique_permissions = len(extract_unique_item(unique_df['permissions']))
    unique_dangerous_permissions = len(extract_unique_item(unique_df['dangerous_permissions']))
    unique_trackers = len(extract_unique_item(unique_df['trackers']))

    total_privacy_policy_yes = (unique_df['is_privacy_policy'] == 'Yes').sum()
    total_privacy_policy_no = (unique_df['is_privacy_policy'] == 'No').sum()

    total_free_apps = (unique_df['free'] == True).sum()
    total_paid_apps = (unique_df['free'] == False).sum()
    total_offers_iap = (unique_df['offersIAP'] == True).sum()
    total_top_grossing = (unique_df['top_grossing'] == 'Yes').sum()

    return {
        'Total Unique Apps Found': total_unique_apps,
        'Total Downloads': total_downloads,
        'Average Downloads': average_downloads,
        'Total Privacy Policy (Yes)': total_privacy_policy_yes,
        'Total Privacy Policy (No)': total_privacy_policy_no,
        'Total Free Apps': total_free_apps,
        'Total Paid Apps': total_paid_apps,
        'Total Apps Offer In-App Purchase': total_offers_iap,
        'Total Top-Grossing Apps': total_top_grossing,
        'Total Unique Permissions': unique_permissions,
        'Total Unique Dangerous Permissions': unique_dangerous_permissions,
        'Total Unique Trackers': unique_trackers,
    }

In [ ]:
# NOTE ON SCOPE: `result_df` is loaded from mhealth_apps_metrics.csv, which
# is 01_privacy_metrics.ipynb's *filtered* output (apps with valid
# ADII/DGI/PCLR and a recognized region/country). The originally collected
# corpus in mhealth_apps_filled.csv is larger (1284 apps); apps lacking
# traffic data or falling outside the tracked country list are dropped
# before reaching this file. Concretely, the full corpus includes 2 paid
# apps, both of which are absent from this filtered set -- so "Total Paid
# Apps" below is 0 for the metrics-eligible subset, not because no paid
# apps were collected. All statistics in this notebook describe the
# metrics-eligible subset (931 apps), not the full collected corpus.
try:
    _full_corpus_path = Path('../data/mhealth_apps_filled.csv')
    _full_corpus_apps = pd.read_csv(_full_corpus_path)['app_id'].nunique()
    print(
        f"Full collected corpus (mhealth_apps_filled.csv): {_full_corpus_apps:,} unique apps\n"
        f"Metrics-eligible subset analyzed below (mhealth_apps_metrics.csv): "
        f"{result_df['app_id'].nunique():,} unique apps\n"
    )
except FileNotFoundError:
    pass

print('Statistics of all apps:')
print('=' * 60)

app_stats = compute_statistics(result_df)
print_stats_block("All apps", app_stats)

print('=' * 60)

In [ ]:
print('Statistics by Region:')
print('=' * 60)

for region in sorted(result_df['region'].dropna().unique()):
    region_df = result_df[result_df['region'] == region]
    region_stats = compute_statistics(region_df)
    print_stats_block(region, region_stats)

print('\n\nStatistics by Country:')
print('=' * 60)

for country_code in sorted(result_df['country'].dropna().unique()):
    country_df = result_df[result_df['country'] == country_code]
    country_name = COUNTRY_LABEL_MAP.get(country_code, country_code.upper())
    country_stats = compute_statistics(country_df)
    print_stats_block(f'{country_name} ({country_code.upper()})', country_stats)

## 4. Category distribution visualization

In [ ]:
required_columns = ['app_id', 'categories']
missing_columns = [col for col in required_columns if col not in result_df.columns]
if missing_columns:
    raise KeyError(f'Missing required columns: {missing_columns}')

category_df = (
    result_df[['app_id', 'categories']]
    .dropna(subset=['categories'])
    .drop_duplicates(subset=['app_id'])
    .copy()
)

# Raw count of apps in each category
category_count = (
    category_df['categories']
    .value_counts()
    .sort_values(ascending=False)
)

# Percentage of apps in each category
category_percentage = (
    category_df['categories']
    .value_counts(normalize=True)
    .sort_values(ascending=False) * 100
)

# Combine count and percentage into one table
category_summary = pd.DataFrame({
    'count': category_count,
    'percentage': category_percentage
})

# top_categories = category_percentage.head(5)
# other_categories = category_percentage.iloc[5:].sum()

# labels = top_categories.index.tolist()
# sizes = top_categories.tolist()

# if other_categories > 0:
#     labels.append('Others')
#     sizes.append(other_categories)

# explode = [0.06] * len(top_categories)
# if other_categories > 0:
#     explode.append(0.01)

In [ ]:
labels = category_percentage.index.tolist()
sizes = category_percentage.tolist()

explode = [0.04] * len(labels)

print("\nCategory summary:")
display(category_summary)

plt.figure(figsize=(7.5, 7.5), dpi=300)

colors = plt.cm.Set3(np.linspace(0, 1, len(sizes)))

# Threshold to decide inside vs outside labeling
THRESHOLD = 4  # percent

wedges, texts, autotexts = plt.pie(
    sizes,
    labels=None,
    autopct=lambda p: f'{p:.1f}%' if p >= THRESHOLD else '',
    startangle=140,
    colors=colors,
    explode=explode,
    pctdistance=0.7,   # slightly closer to center
    labeldistance=1.05,
    shadow=False,
    wedgeprops={
        'linewidth': 0.8,
        'edgecolor': 'white',
        'alpha': 0.9
    },
    textprops={'fontsize': 12}
)

# Style wedges (shadow)
for w in wedges:
    w.set_path_effects([
        pe.SimplePatchShadow(
            offset=(2, -2),
            shadow_rgbFace=(0.85, 0.85, 0.85),
            alpha=0.6
        ),
        pe.Normal()
    ])

# Style inside labels
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_alpha(0.9)
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11.5)

# Add OUTSIDE labels for small slices (no overlap)
for i, (wedge, size) in enumerate(zip(wedges, sizes)):
    if size < THRESHOLD:
        angle = (wedge.theta2 + wedge.theta1) / 2
        x = np.cos(np.deg2rad(angle))
        y = np.sin(np.deg2rad(angle))

        plt.annotate(
            f"{size:.1f}%",
            xy=(x * 0.9, y * 0.9),          # start at wedge
            xytext=(x * 1.15, y * 1.15),    # outside position
            ha='center',
            va='center',
            fontsize=12,
            fontweight='bold',
            arrowprops=dict(
                arrowstyle='-',
                color='gray',
                lw=0.8
            )
        )

# Legend
plt.legend(
    labels,
    title='Categories',
    loc='center left',
    bbox_to_anchor=(0.95, 0.5),
    fontsize=11,
    title_fontsize=12,
    frameon=False
)

plt.tight_layout()
plt.savefig(FIG_DIR / 'apps_by_category.png', dpi=600, bbox_inches='tight')
plt.show()